# 第25章　実践 ― CT膵がん検出パイプラインをE2Eで組み上げる**『本格実装 医療診断支援AI（実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-impl

## 25.3　前処理 ― ばらばらのデータをそろえる

In [ ]:
from monai import transforms as T# この前処理はMONAIで学習する経路用。nnU-Netには適用しない# （間隔の統一とCT正規化はnnU-Net側が自前で行う。変換前のNIfTIを渡す）pre = T.Compose([    T.LoadImaged(keys=["image", "label"]),               # NIfTIを読む    T.EnsureChannelFirstd(keys=["image", "label"]),      # 先頭にチャネル軸を足す（空間軸の順はファイルのまま）    T.Orientationd(keys=["image", "label"], axcodes="RAS"),   # 向きを統一    T.Spacingd(keys=["image", "label"],                  # ボクセル間隔を揃える               pixdim=(1.5, 1.5, 1.5),               mode=("bilinear", "nearest")),            # 画像は補間、ラベルは最近傍    T.ScaleIntensityRanged(keys="image",                 # HUクリップ＋0〜1正規化               a_min=-100, a_max=240, b_min=0.0, b_max=1.0, clip=True),    T.CropForegroundd(keys=["image", "label"], source_key="image"),  # 空気の余白を削る])

In [ ]:
import numpy as np, nibabel as nib, cc3ddef audit_case(case_id, label_path, tumor_id=18):    img = nib.load(label_path)    lab = img.get_fdata().astype(np.uint8)    # spacing は必ず実データのヘッダから読む。既定値1.5を当てると実寸ごと狂う。    sx, sy, sz = img.header.get_zooms()[:3]    vox_mm3 = float(sx) * float(sy) * float(sz)    comps = cc3d.connected_components(lab == tumor_id)   # 腫瘍の塊ごとに分ける    rows = []                                            # 1病変1行で残す    for cid in range(1, comps.max() + 1):        vol = float((comps == cid).sum()) * vox_mm3        # 体積から求めた「等価球の直径」。臨床でいう最大径（軸位断の最長径。RECISTの流儀）        # とは別物で、細長い腫瘍では過小に出る。だからキー名で区別しておく。        rows.append({"case_id": case_id, "lesion_id": cid, "class": tumor_id,                     "volume_mm3": vol,                     "equivalent_sphere_diameter_mm": 2 * ((3*vol/(4*np.pi)) ** (1/3))})    return rows        # 症例の代表値が要るときは、この表から後で作る

## 25.4　データを分ける ― 層化分割とnnU-Net形式

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold# 前節の audit_case は「1病変1行」を返す。層化はそれを症例ごとに畳んで作る。# lesions[case_id] = その症例の病変行のリスト（病変が無ければ空）def stratum(rows):    if not rows:                                                   return "none"    if max(r["equivalent_sphere_diameter_mm"] for r in rows) <= 10: return "small"    return "large"strata = [stratum(lesions[c]) for c in case_ids]# 注意：ここで層化に使うのは「等価球直径」であって、臨床の最長径（軸位断の最長径。RECISTの# 流儀）とは別物で、細長い腫瘍では過小に出る。性能表に「10mm以下」と書くときは、どちらの径で# 切ったのかを必ず併記すること。# 同じ患者の別検査・別相が train と val に割れないよう、患者IDでグループ化して層化する。# StratifiedKFold（case単位）では、多相CTや再検査を持つ患者がまたいで漏れる。patient_ids = [manifest[c]["patient_id"] for c in case_ids]  # case→patient はmanifestで固定sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)folds = list(sgkf.split(case_ids, strata, groups=patient_ids))for tr, va in folds:                               # 患者単位で切れているかを必ず検算する    assert not ({patient_ids[i] for i in tr} & {patient_ids[i] for i in va})

In [ ]:
import json, os# nnU-Netのcase識別子は labelsTr のファイル名から拡張子を除いたもの（画像側の _0000 は付けない）splits = [{"train": [case_ids[i] for i in tr], "val": [case_ids[i] for i in va]}          for tr, va in folds]out = os.path.join(os.environ["nnUNet_preprocessed"], "Dataset007_Pancreas")json.dump(splits, open(os.path.join(out, "splits_final.json"), "w"), indent=2)

```bash# 実装編の実行環境（WSL2＋Docker）に合わせて設定。入門編の Colab パスとは異なる。export nnUNet_raw="/workspace/nnUNet_raw"export nnUNet_preprocessed="/workspace/nnUNet_preprocessed"export nnUNet_results="/workspace/nnUNet_results"```

In [ ]:
import jsondataset = {    "channel_names": {"0": "CT"},    "labels": {"background": 0, "pancreas": 1, "tumor": 2},    "numTraining": 281,    "file_ending": ".nii.gz",}json.dump(dataset, open("nnUNet_raw/Dataset007_Pancreas/dataset.json", "w"), indent=2)

## 25.5　nnU-Netのカスケードで学習する

```bash# ① データを検証し、前処理・設計を自動生成nnUNetv2_plan_and_preprocess -d 7 --verify_dataset_integrity# ② カスケードの第1段（低解像度）を5分割ぶん学習for f in 0 1 2 3 4; do nnUNetv2_train 7 3d_lowres $f; done# ③ 第2段（第1段の出力を手がかりにフル解像度で精密化）for f in 0 1 2 3 4; do nnUNetv2_train 7 3d_cascade_fullres $f; done```

## 25.7　推論 ― スライディングウィンドウで1症例を処理する

```bash# カスケードは2段階。第1段(低解像度)の予測を、第2段に渡す必要がある。# -prev_stage_predictions を忘れると、第2段はエラーで止まる。nnUNetv2_predict -i imagesTs/ -o pred_lowres/ -d 7 \    -c 3d_lowres -f 0 1 2 3 4nnUNetv2_predict -i imagesTs/ -o predictions/ -d 7 \    -c 3d_cascade_fullres -f 0 1 2 3 4 \    -prev_stage_predictions pred_lowres/    # ← 第1段の出力を手がかりにする```

In [ ]:
from monai.inferers import sliding_window_inferencemodel.eval()with torch.no_grad():                                   # 推論では勾配を追跡しない（基礎編のPyTorchの章）    logits = sliding_window_inference(        inputs=ct_volume,          # (1, 1, D, H, W)        roi_size=(96, 160, 160),   # パッチの大きさ        sw_batch_size=2,        predictor=model,        overlap=0.5,               # パッチを50%重ねて継ぎ目を滑らかに    )    pred = logits.argmax(dim=1)    # 各ボクセルを最も確率の高いクラスへ

## 25.8　評価 ― Diceだけでは、膵がんは測れない

In [ ]:
import cc3d, numpy as npfrom scipy.sparse import csr_matrixfrom scipy.sparse.csgraph import maximum_bipartite_matchingdef lesion_detection(pred_mask, gt_mask, overlap_thr=0.1):  # 重なり=GT被覆率（IoUではない）    # 対応づけは1対1。1つの予測塊を複数のGTに使い回すと、全面塗りつぶしが感度1.00になる。    # それでも塗りすぎは完全には罰せられないので、Diceと予測総体積を必ず並べて読むこと。    pred_cc = cc3d.connected_components(pred_mask)    gt_cc   = cc3d.connected_components(gt_mask)    n_pred, n_gt = int(pred_cc.max()), int(gt_cc.max())    # ① 候補の辺：正解 g を予測塊 p が被覆率（交わり ÷ 正解のボクセル数）overlap_thr 以上で覆う    rows, cols = [], []    for g in range(1, n_gt + 1):        gt = (gt_cc == g); gt_n = float(gt.sum())        for p in range(1, n_pred + 1):            if float(((pred_cc == p) & gt).sum()) / gt_n >= overlap_thr:                rows.append(g - 1); cols.append(p - 1)    # ② 1対1の対応数を最大化（二部グラフの最大マッチング）。正解を番号順に見て貪欲に確保すると、    #    先の正解が別の正解の唯一の候補を奪い、同じ配置を左右反転しただけでTPが変わる。    #    最大マッチングなら対応数は走査順に依存しない（基礎編の評価の章と同じ実装）    tp = 0    if rows:        adj = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(n_gt, n_pred))        tp = int((maximum_bipartite_matching(adj, perm_type="column") >= 0).sum())    fp = n_pred - tp                            # どのGTにも対応づかなかった予測塊＝偽陽性    # 病変が無い症例の検出率は0ではなく「測れない」（基礎編の評価の章と同じ規約）。    # 陰性症例は感度ではなく、症例あたり偽陽性数（FP/scan）と症例特異度で評価する。    return {"tp": tp, "fp": fp, "n_gt": n_gt, "n_pred": n_pred,            "lesion_recall":    tp / n_gt   if n_gt   else float("nan"),            "lesion_precision": tp / n_pred if n_pred else float("nan"),            "pred_voxels": int(pred_mask.sum()), "gt_voxels": int(gt_mask.sum())}

## 間接所見を測る ― 主膵管と胆管の拡張を数値化する

In [ ]:
import numpy as npfrom skimage.morphology import skeletonizefrom scipy.ndimage import distance_transform_edt# 判定の規則は関数に埋め込まず、基準名・対象・適用外を持つ「規則」として渡す（24.10節の FLR と同じ設計）MPD_RULE = {"name": "MPD-body", "duct": "主膵管（体部）", "dilate_thr_mm": 3.0,            "applies_to": "成人・造影門脈相", "not_for": "術後変化・慢性膵炎既知例は別途判断"}CBD_RULE = {"name": "CBD", "duct": "総胆管", "dilate_thr_mm": 7.0,            "applies_to": "胆嚢摘出歴なし", "not_for": "胆嚢摘出後・高齢者は 10.0 mm の別規則を使う"}def duct_caliber(duct_mask, mm_per_vox, rule, span_mm=5.0, drop_mm=2.0):    """戻り値の abrupt_cutoff は True / False / None の3値。None は「評価不能」で、state に理由を書く。    分断・分岐・短すぎる中心線を False（途絶なし）にしてはいけない ― 最も注意すべき分断が「正常」に化ける。"""    NA = {"max_diameter_mm": None, "dilated": None, "abrupt_cutoff": None, "rule": rule["name"]}    mm_per_vox = np.asarray(mm_per_vox, dtype=float)    if mm_per_vox.shape != (3,) or np.any(~np.isfinite(mm_per_vox)) or np.any(mm_per_vox <= 0):        return {**NA, "state": "評価不能（ボクセル間隔が不正）"}    if duct_mask.sum() == 0:        return {**NA, "state": "評価不能（管のマスクが空）"}    # EDTは「最近傍の背景ボクセルの中心までの距離」なので、境界までの距離より大きく出る。    # 半ボクセルを引くのは近似で、方向や形状によって過不足が残る。3mm前後の管を1.5mm格子で    # 測るときは、この誤差が閾値判定を直撃する。あくまで「距離変換由来の近似径」として扱う。    dist   = distance_transform_edt(duct_mask, sampling=mm_per_vox) - 0.5 * float(mm_per_vox.min())    center = skeletonize(duct_mask) > 0                      # 中心線（skeletonizeは3Dも自動処理）    pts    = np.argwhere(center)                             # 中心線のボクセル座標    if len(pts) < 2:        return {**NA, "state": "評価不能（中心線が取れない）"}    diam   = 2.0 * np.clip(dist[center], 0.0, None)          # 直径[mm]（順序は pts と対応）    max_d  = float(diam.max())    result = {"max_diameter_mm": round(max_d, 1),              "dilated": bool(max_d > rule["dilate_thr_mm"]),  # 規則に応じた閾値              "rule": rule["name"]}    # 「急な口径の途絶（abrupt cutoff）」は、中心線に沿った“並び順”でしか見えない。    # 径を大きさでソートすると空間的な並びが壊れ、段差は永久に検出できなくなる。    # そこで、端点から測地距離順に中心線をたどり、その順で径のプロファイルを作る。    trace = order_along_centerline(pts, mm_per_vox)          # 添字列・区間長・分断/分岐の情報    order, seg_len = trace["order"], trace["seg_len_mm"]    arc = np.concatenate([[0.0], np.cumsum(seg_len)])        # 中心線に沿った累積弧長[mm]    result.update({"centerline_points": int(len(pts)), "visited_points": int(len(order)),                   "n_components": trace["n_components"], "has_branch": trace["has_branch"],                   "measured_length_mm": round(float(arc[-1]), 1)})    # 途絶の判定ができるのは、中心線が一続き（分断なし）で、分岐がなく、span_mm 以上たどれたときだけ    if trace["n_components"] != 1 or trace["has_branch"]:        return {**result, "abrupt_cutoff": None,                "state": "評価不能（中心線が分断または分岐。部分計測のみ）"}    if arc[-1] < span_mm:        return {**result, "abrupt_cutoff": None,                "state": f"評価不能（たどれた長さ {arc[-1]:.1f} mm が {span_mm} mm 未満）"}    # 途絶の定義：中心線上の各点 i と、弧長で span_mm 以上先にある最初の点 j（したがって評価区間は    # span_mm 以上、span_mm＋1ボクセル対角長未満）との径の差の絶対値が drop_mm 以上。    # 補間はせず離散点の値で比べる。片方の端から始めるので上流・下流の向きは保証されず、    # 「細くなる」向きに限らず両向きの変化を拾う（急な口径変化としての途絶）。    # 「何点ぶん離れているか」で数えてはいけない。26近傍では1点進むだけでも対角方向なら    # 1.41倍・1.73倍の距離を動くので、点数は物理距離を保証しない。    profile = diam[order]                                    # 中心線に沿った径の列    drop, j = 0.0, 0    for i in range(profile.size):        while j < profile.size and arc[j] - arc[i] < span_mm:   # 弧長で span_mm 以上先を探す            j += 1        if j >= profile.size:            break        drop = max(drop, abs(float(profile[j] - profile[i])))    return {**result, "abrupt_cutoff": bool(drop >= drop_mm),            "max_drop_mm": round(drop, 1), "state": "ok"}def order_along_centerline(pts, mm_per_vox):    """中心線のボクセルを片方の端点から順にたどる。戻り値は辞書：    order（たどれた添字列）、seg_len_mm（各区間の物理長[mm]）、n_components（中心線の連結成分数）、    has_branch（たどる途中で進路が2つ以上に分かれた）。分断や分岐があるときは、呼び出し側が    「部分計測」として扱い、途絶の有無を判定してはいけない。"""    from scipy.spatial import cKDTree    tree = cKDTree(pts)    nbr = tree.query_ball_point(pts, r=np.sqrt(3) + 1e-6)     # 26近傍でつながる点（自分自身を含む）    deg = np.array([len(n) - 1 for n in nbr])                 # 自分自身を除いた隣接数    # 連結成分の数：26近傍でつながる点の集まりを数える    comp, n_components = -np.ones(len(pts), dtype=int), 0    for s0 in range(len(pts)):        if comp[s0] >= 0:            continue        stack, comp[s0] = [s0], n_components        while stack:            u = stack.pop()            for v in nbr[u]:                if comp[v] < 0:                    comp[v] = n_components; stack.append(v)        n_components += 1    start = int(np.argmin(np.where(deg == 1, 0, deg + 1)))    # deg==1 の端点を優先    order, seen, cur, has_branch = [start], {start}, start, False    while True:        nxt = [j for j in nbr[cur] if j not in seen]        if not nxt:            break        if len(nxt) > 1:            has_branch = True                                 # 進路が分かれた。以後の順序は一意でない        cur = nxt[0]        seen.add(cur)        order.append(cur)    order = np.array(order)    # 隣り合う点の物理距離[mm]。点数ではなく、この長さで区間を測る。    d = (pts[order[1:]] - pts[order[:-1]]) * np.asarray(mm_per_vox, dtype=float)    return {"order": order, "seg_len_mm": np.linalg.norm(d, axis=1),            "n_components": int(n_components), "has_branch": bool(has_branch)}# 使い方：result = duct_caliber(mpd_mask, spacing, MPD_RULE)#   result["abrupt_cutoff"] は None（評価不能）を False（途絶なし）と同じに扱わない。#   合成した中心線（分断した5点、分岐した形、4 mm しかない短い管）で None が返ることを確かめてから使う

## 25.10　アンサンブルとTTA ― 最後のひと押し

In [ ]:
def tta_predict(model, volume, tta_axes):    """tta_axes は学習時の変換一覧と同じ設定から生成する（モデル設定に保存しておく）。    どの軸の反転を許すかは、部位・タスク・左右ラベルの有無・学習時の拡張で決まる。    たとえば頭尾方向の反転は、腹部CTでは採らない設計もありうる。検証データで確かめること。"""    preds = []    for axes in tta_axes:                        # 例: [(), (3,), (4,)]（頭尾は入れない）        flipped = torch.flip(volume, dims=axes) if axes else volume        with torch.no_grad():            out = sliding_window_inference(flipped, (96,160,160), 2, model)        preds.append(torch.flip(out, dims=axes) if axes else out)  # 元に戻す    # ここで平均しているのは softmax 前のロジット。確率を平均する実装とは結果が変わるので、    # どちらの方式か明示し、較正と閾値選択はその最終方式のうえで行う。    return torch.stack(preds).mean(0)            # 平均して安定化